In [ ]:
import pandas as pd
import csv
from openai import OpenAI
import time
import json

client = OpenAI(api_key="")

def extract_keywords_3d(row):
    undl_id = row["undl_id"]
    title = row["title"]
    
    prompt = f"""
You are a data expert proficient in international relations. Your task is to extract core keywords from the provided UN resolution title across three distinct dimensions: Geopolitical, Thematic, and Action. The extraction must be strict and precise.

Task Description:
Analyze the title and extract the following three categories of information:
1. Geopolitical: Identify specific countries, regions, or territories mentioned (e.g., Ukraine, Syria, Palestine). If no specific geographical entity is present, leave this field empty.
2. Thematic: Pinpoint the core subject matter or technical domain (e.g., Human Rights, Digital Technologies, Terrorism).
3. Action: Determine the diplomatic means, objectives, or verbs indicating a course of action (e.g., International Cooperation, Investigation, Promotion).

Extraction Rules:
- Strict Extraction with Noun Normalization: All extracted keywords MUST originate from the original title text. However, you MUST normalize all extracted terms into nouns or noun phrases. For example, convert verbs or gerunds into their noun forms (e.g., "Improving" -> "Improvement", "Promoting" -> "Promotion").
- Semantic Aggregation (CRITICAL): Keep proper nouns and established phrases strictly intact. Do NOT split phrases like "National or Ethnic, Religious and Linguistic Minorities". It must remain as a single, unified phrase. Do NOT split by "or", "and", or commas if they are part of a single established concept or title.
- Entity Normalization: Simplify names (e.g., "territories of Ukraine" -> "Ukraine").
- Thorough Denoising: You MUST strictly delete ALL administrative, formatting, and meaningless words. This includes, but is not limited to: "resolution", "adopted by", "General Assembly", "A/RES/", "Situation in", "Situation of", "field of", "context of", "Work of"...
- Multi-value Handling: If a category yields multiple distinct keywords (not parts of a single phrase), separate them with a semicolon (`;`). Do NOT use commas, as they interfere with CSV structure. If a category has no relevant keywords, leave its field empty.

Output Requirements:
Return the result strictly as a JSON object with the following keys:
- "Geopolitical": string (semicolon-separated or empty)
- "Thematic": string (semicolon-separated or empty)
- "Action": string (semicolon-separated or empty)

Title to analyze:
{title}
"""
    try:
        response = client.chat.completions.create(
            model="gpt-4.1-mini",
            messages=[
                {"role": "system", "content": "You are a data expert proficient in international relations. Strictly adhere to the extraction rules, especially regarding semantic aggregation (do not split established phrases) and noun normalization."},
                {"role": "user", "content": prompt}
            ],
            response_format={"type": "json_object"},
            temperature=0
        )
        content = response.choices[0].message.content.strip()
        data = json.loads(content)
        
        return {
            "Original_ID": undl_id,
            "Geopolitical": data.get("Geopolitical", ""),
            "Thematic": data.get("Thematic", ""),
            "Action": data.get("Action", "")
        }
    except Exception as e:
        print(f"Error processing {undl_id}: {e}")
        return {
            "Original_ID": undl_id,
            "Geopolitical": "",
            "Thematic": "",
            "Action": ""
        }
import os

def main():
    input_file = 'E:/ETH/UN/code/policy-pulse/data/resolution_titles.csv'
    output_file = 'undlid_keywords_3d_noun_fixed.csv'
    
    df = pd.read_csv(input_file)[30:]
    total = len(df)
    print(f"Processing {total} rows...")

    # Check if file exists (to avoid rewriting header)
    file_exists = os.path.isfile(output_file)

    with open(output_file, mode='a', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(
            f,
            fieldnames=["Original_ID", "Geopolitical", "Thematic", "Action"]
        )

        # Write header only once
        if not file_exists:
            writer.writeheader()

        for i, row in df.iterrows():
            result = extract_keywords_3d(row)

            # Write immediately
            writer.writerow(result)
            f.flush()  # 🔥 forces write to disk instantly

            print(f"Processed {i + 1}/{total} rows...")
            time.sleep(0.5)

    print(f"Saved results to {output_file}")

# def main():
#     input_file = 'E:/ETH/UN/code/policy-pulse/data/resolution_titles.csv'
#     output_file = 'undlid_keywords_3d_noun_fixed.csv'
    
#     df = pd.read_csv(input_file)[:30]
#     total = len(df)
#     print(f"Processing {total} rows...")
    
#     results = []
#     for i, row in df.iterrows():
#         result = extract_keywords_3d(row)
#         results.append(result)
#         print(f"Processed {i + 1}/{total} rows...")
#         time.sleep(0.5) # Small delay to avoid rate limits
            
#     output_df = pd.DataFrame(results)
#     output_df = output_df[["Original_ID", "Geopolitical", "Thematic", "Action"]]
#     # Use standard CSV writing to avoid the extra empty columns issue seen previously
#     output_df.to_csv(output_file, index=False)
#     print(f"Saved results to {output_file}")

if __name__ == "__main__":
    main()


Processing 5664 rows...
Processed 31/5664 rows...
Processed 32/5664 rows...
Processed 33/5664 rows...
Processed 34/5664 rows...
Processed 35/5664 rows...
Processed 36/5664 rows...
Processed 37/5664 rows...
Processed 38/5664 rows...
Processed 39/5664 rows...
Processed 40/5664 rows...
Processed 41/5664 rows...
Processed 42/5664 rows...
Processed 43/5664 rows...
Processed 44/5664 rows...
Processed 45/5664 rows...
Processed 46/5664 rows...
Processed 47/5664 rows...
Processed 48/5664 rows...
Processed 49/5664 rows...
Processed 50/5664 rows...
Processed 51/5664 rows...
Processed 52/5664 rows...
Processed 53/5664 rows...
Processed 54/5664 rows...
Processed 55/5664 rows...
Processed 56/5664 rows...
Processed 57/5664 rows...
Processed 58/5664 rows...
Processed 59/5664 rows...
Processed 60/5664 rows...
Processed 61/5664 rows...
Processed 62/5664 rows...
Processed 63/5664 rows...
Processed 64/5664 rows...
Processed 65/5664 rows...
Processed 66/5664 rows...
Processed 67/5664 rows...
Processed 68/5